# 08 - Kaggle EnMAP L2A Dataset Downloader & Packager (STAC Spectral Cube)

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook queries the **DLR EOC STAC API** to discover real EnMAP Level-2A hyperspectral scenes over the Liwa Oasis AOI (`[53.5, 22.8, 54.2, 23.3]`) and downloads full multi-band spectral data cubes (`SPECTRAL_IMAGE.TIF`, >100 MB) into `/kaggle/working/enmap_liwa_dataset`.

In [ ]:
# Automatic dependency check & installation for Kaggle/Colab/Local environments
# The mechanistic justification for this step is: Dynamically inspecting and installing required packages (planetary-computer, pystac-client, rasterio, geopandas, folium, h5py) prevents runtime ModuleNotFoundError exceptions across Kaggle runtime environments.
import subprocess
import sys

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("planetary_computer", "planetary-computer")
install_if_missing("pystac_client", "pystac-client")
install_if_missing("pystac", "pystac")
install_if_missing("rasterio", "rasterio")
install_if_missing("geopandas", "geopandas")
install_if_missing("folium", "folium")
install_if_missing("h5py", "h5py")

# Step 1: Initialize Environment & Directory
import os
import json
import getpass
import requests
from tqdm import tqdm

DATASET_DIR = "/kaggle/working/enmap_liwa_dataset"
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Target dataset output directory: {DATASET_DIR}")


In [ ]:
# Step 2: Configure DLR Credentials
import os

DLR_USERNAME = os.environ.get("DLR_USERNAME", "")
DLR_PASSWORD = os.environ.get("DLR_PASSWORD", "")
if not DLR_USERNAME or not DLR_PASSWORD:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    DLR_USERNAME = secrets.get_secret("DLR_USERNAME")
    DLR_PASSWORD = secrets.get_secret("DLR_PASSWORD")

print(f"Configured credentials for DLR user: {DLR_USERNAME}")


In [ ]:
# Step 3: Query DLR STAC API & Target Full Hyperspectral Cubes
STAC_SEARCH_URL = "https://geoservice.dlr.de/eoc/ogc/stac/v1/search"
LIWA_BBOX = [53.5, 22.8, 54.2, 23.3]

stac_query = {
    "collections": ["ENMAP_HSI_L2A"],
    "bbox": LIWA_BBOX,
    "limit": 5
}

def select_spectral_asset(assets):
    """Extracts the true multi-band spectral cube, excluding QuickLook previews."""
    # 1. Direct STAC standard key lookup
    for key in ["spectral", "SPECTRAL_IMAGE", "SPECTRAL_IMAGE_COG", "product", "data"]:
        if key in assets:
            href = assets[key].get("href", "")
            if href and "QL_" not in href and "pixelmask" not in href.lower():
                return href
    
    # 2. Search by filename matching SPECTRAL_IMAGE
    for k, v in assets.items():
        if isinstance(v, dict):
            href = v.get("href", "")
            if ("SPECTRAL_IMAGE" in href or "spectral" in k.lower()) and "QL_" not in href and "pixelmask" not in href.lower():
                return href

    # 3. Fallback: Any TIF that is NOT quicklook/pixelmask/quality
    for k, v in assets.items():
        if isinstance(v, dict):
            href = v.get("href", "")
            if href.endswith(".TIF") and not any(ign in href for ign in ["QL_", "PIXELMASK", "QUALITY", "PREVIEW"]):
                return href
    return None

def select_metadata_asset(assets):
    for key in ["metadata", "METADATA"]:
        if key in assets:
            return assets[key].get("href")
    for k, v in assets.items():
        if isinstance(v, dict) and v.get("href", "").endswith(".xml"):
            return v.get("href")
    return None

print(f"Querying DLR STAC API at {STAC_SEARCH_URL}...")
resp = requests.post(STAC_SEARCH_URL, json=stac_query, headers={"Content-Type": "application/json"})

scene_targets = []
if resp.status_code == 200:
    features = resp.json().get("features", [])
    print(f"Found {len(features)} EnMAP L2A scene(s) matching Liwa Oasis AOI.")
    for f in features:
        scene_id = f.get("id")
        assets = f.get("assets", {})
        
        tif_href = select_spectral_asset(assets)
        xml_href = select_metadata_asset(assets)
        
        if tif_href:
            scene_targets.append({
                "id": scene_id,
                "tif_url": tif_href,
                "xml_url": xml_href
            })
            print(f"  - Scene: {scene_id}")
            print(f"    Target Spectral Cube: {tif_href}")
else:
    print(f"STAC search returned HTTP {resp.status_code}: {resp.text[:300]}")


In [ ]:
# Step 4: Stream Download Scene Assets with Basic Authentication
def download_asset(url, target_file, username, password):
    if not url:
        return False
    print(f"Downloading {os.path.basename(target_file)} from {url}...")
    session = requests.Session()
    session.auth = (username, password)
    
    res = session.get(url, stream=True)
    if res.status_code == 200:
        content_type = res.headers.get('content-type', '').lower()
        if 'html' in content_type:
            print(f"[ERROR] HTTP 200 returned HTML login page. Check credentials for user '{username}'.")
            return False
        
        total_bytes = int(res.headers.get('content-length', 0))
        with open(target_file, 'wb') as f, tqdm(total=total_bytes, unit='B', unit_scale=True) as pbar:
            for chunk in res.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
        
        mb_size = os.path.getsize(target_file) / (1024 * 1024)
        print(f"[SUCCESS] Saved {target_file} ({mb_size:.2f} MB)")
        return True
    else:
        print(f"[ERROR] Failed to download {url} (HTTP {res.status_code})")
        return False

for target in scene_targets:
    s_dir = os.path.join(DATASET_DIR, target["id"])
    os.makedirs(s_dir, exist_ok=True)
    
    t_path = os.path.join(s_dir, "SPECTRAL_IMAGE.TIF")
    x_path = os.path.join(s_dir, "METADATA.xml")
    
    download_asset(target["tif_url"], t_path, DLR_USERNAME, DLR_PASSWORD)
    if target["xml_url"]:
        download_asset(target["xml_url"], x_path, DLR_USERNAME, DLR_PASSWORD)


In [ ]:
# Step 5: Write Kaggle Dataset Metadata & Verify Files
meta = {
    "title": "EnMAP Hyperspectral Liwa Oasis L2A Dataset",
    "id": "assemelqersh/enmap-liwa-l2a",
    "licenses": [{"name": "CC-BY-4.0"}]
}

meta_file = os.path.join(DATASET_DIR, "dataset-metadata.json")
with open(meta_file, "w") as f:
    json.dump(meta, f, indent=2)

print("=== Downloaded Dataset Verification ===")
for r, d, files in os.walk(DATASET_DIR):
    for fl in files:
        fp = os.path.join(r, fl)
        sz = os.path.getsize(fp) / (1024 * 1024)
        print(f"{fp} ({sz:.2f} MB)")
